# Phase 1 demo — data loading and model + LoRA target enumeration

This notebook exercises everything implemented so far:

- **Phase 1.1** — [`src/data.py`](../src/data.py): `set_seed`, `load_sst2`, `make_dataloaders`.
- **Phase 1.2** (Unit 1.2.1) — [`src/models.py`](../src/models.py): `load_model_and_tokenizer`, `find_lora_target_module_names`, `module_dims`, `count_parameters`.

It ends with a forward pass through DistilBERT on a real SST-2 batch, so you can see all five primitives line up end-to-end before any LoRA / training code lands.

**Run from the repo root** so `from src.data import ...` resolves. From this `notebooks/` directory, JupyterLab will already have the right cwd if you launched `jupyter lab` from the repo root.

**Prerequisites:** `.venv` activated; `pip install -r requirements.txt` plus `pip install jupyter` (Jupyter isn't in `requirements.txt` since it's only for ad-hoc demos).

## 0. Setup

In [ ]:
# Make sure the repo root is on sys.path so `from src...` works,
# regardless of where the notebook server was started.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

In [ ]:
import torch

from src.data import load_sst2, make_dataloaders, set_seed
from src.models import (
    count_parameters,
    find_lora_target_module_names,
    load_model_and_tokenizer,
    module_dims,
)

set_seed(42)
print(f"torch {torch.__version__}  |  cuda available: {torch.cuda.is_available()}")

## 1. Load the model and tokenizer (Phase 1.2)

First call downloads ~250 MB of DistilBERT weights into `~/.cache/huggingface/`. Subsequent calls reuse the cache.

In [ ]:
model, tokenizer = load_model_and_tokenizer("distilbert-base-uncased", num_labels=2)
print(type(model).__name__, "|", type(tokenizer).__name__)

## 2. Enumerate the 12 LoRA target modules

DistilBERT base has 6 transformer layers. Each layer's attention exposes `q_lin` and `v_lin` linear projections. That's 12 LoRA targets, all 768 → 768. The project's rank-budget invariant `total_rank_budget = 96` divides cleanly: 96 / 12 = 8 ranks per module under uniform allocation.

The cost column previews what the Phase 4b allocator will use as its denominator: `c_i = in_features + out_features`.

In [ ]:
target_names = find_lora_target_module_names(model)
print(f"Found {len(target_names)} LoRA target modules\n")
print(f"{'fully-qualified name':<60s}  {'in':>4} -> {'out':<4}  {'cost (d+k)':>10}")
print("-" * 88)
for n in target_names:
    in_dim, out_dim = module_dims(model, n)
    print(f"{n:<60s}  {in_dim:>4} -> {out_dim:<4}  {in_dim + out_dim:>10}")

## 3. Parameter counts

On a fresh HF model nothing is frozen, so `trainable == total`. Once PEFT wraps the model in Phase 4a, the trainable count will drop sharply (only LoRA A/B matrices remain trainable).

In [ ]:
total = count_parameters(model, trainable_only=False)
trainable = count_parameters(model, trainable_only=True)
print(f"Total parameters:     {total:>12,}")
print(f"Trainable (pre-PEFT): {trainable:>12,}")
print("  -> equal because no LoRA has been attached yet")

## 4. Load SST-2 (Phase 1.1)

We use a small subset for the demo so the cell runs in seconds. Production runs use the full splits (~67k train / 872 val).

In [ ]:
splits = load_sst2(
    tokenizer,
    max_length=128,
    max_train_samples=1024,
    max_val_samples=128,
)
print(f"Train: {len(splits.train):>5d} examples")
print(f"Val:   {len(splits.val):>5d} examples")
print(f"Labels: {splits.num_labels}")
print("\nFirst train example fields:", list(splits.train[0].keys()))

In [ ]:
train_loader, val_loader = make_dataloaders(
    splits, tokenizer, batch_size=16, num_workers=0
)
batch = next(iter(train_loader))
print("Batch keys:", list(batch.keys()))
for k, v in batch.items():
    print(f"  {k}: shape={tuple(v.shape)}, dtype={v.dtype}")

## 5. End-to-end forward pass

Tokenizer + dataloader + model wired together. The untrained classifier head should land near 50% accuracy on a 2-class task (chance), confirming the plumbing works without bias toward either label.

In [ ]:
model.eval()
with torch.no_grad():
    inputs = {k: v for k, v in batch.items() if k != "labels"}
    out = model(**inputs)

logits = out.logits
preds = logits.argmax(dim=-1)
labels = batch["labels"]
acc = (preds == labels).float().mean().item()

print(f"Logits shape: {tuple(logits.shape)}  (batch_size, num_labels)")
print(f"Predictions:  {preds.tolist()}")
print(f"Labels:       {labels.tolist()}")
print(f"\nBatch accuracy on the untrained model: {acc:.3f}  (~0.5 expected for 2-class chance)")

## 6. Preview: the allocator's cost vector

Phase 4b's `HardwareAwareRankAllocator` will use `c_i = in_features + out_features` as the per-rank cost in the score `s_i = g_i / c_i^α`. For DistilBERT base every target is 768×768 so all costs are 1536 and α has nothing to penalize — the *interesting* case for hardware-aware allocation needs heterogeneous module sizes (e.g. extending to MLP up/down projections, which are 3072-wide). We're keeping the q/v-only setup for fair comparison with the four configs; that test will come from a later α ablation if we extend the target set.

In [ ]:
costs = {n: sum(module_dims(model, n)) for n in target_names}
unique_costs = sorted(set(costs.values()))
print(f"Unique cost values across the 12 targets: {unique_costs}")
print(f"Total cost (Σ c_i):                        {sum(costs.values())}")

## What's next

- **Unit 2.1** — `src/hardware_logger.py`: JSONL log writer + wall-clock / memory / scheduler-overhead probes.
- **Unit 3.1** — `src/evaluate.py`: validation loop and target-accuracy tracker.
- **Unit 4a.1 / 4b.1** — LoRA enumeration + the rank allocator (the project's contribution).
- **Unit 5.x** — `src/train.py` for all four methods.

Re-run this notebook after each subsequent unit lands to watch the progress: by Unit 4a.1 the model in cell 1 will be a `PeftModel` and the trainable parameter count in cell 3 will plummet.